# Qwen2.5-0.5B-Instruct — Concept Probe Robustness (20 pairs)

## Purpose
Re-evaluate the X-vs-Y Concept Probe with the thesis model, using held-out descriptions that differ in features and contexts from the training descriptions.

This notebook does **not** measure whether a concept is absolutely present or absent. The Probe predicts which of two concepts (X or Y) is more consistent with a hidden state.

## Fixed design
- Model: `Qwen/Qwen2.5-0.5B-Instruct` (pinned revision)
- 20 concept pairs / 40 concepts
- Training: 6 description families per concept × 5 surface frames
- Evaluation: 5 newly authored description families per concept × 4 surface frames
- Instructions at evaluation: neutral, mention X, think X, avoid X, think Y
- Layers: 22, 23, 24
- Readout positions: end of context and assistant-generation start
- Probe: StandardScaler + LogisticRegression (`C=0.1`)
- Training stability: six leave-one-training-family-out folds
- Negative control: all 252 balanced assignments of labels to the 10 training families in each fold
- Evaluation robustness: recompute AUC after removing each matched X/Y evaluation-family pair

## Important interpretation limits
1. Surface frames are repeated renderings of one semantic description family; they are not independent semantic examples.
2. The six training folds overlap. They are sensitivity checks, not six independent replications.
3. AUC is calculated primarily over family-averaged scores, not by treating every wrapper as independent.
4. The permutation control shuffles labels by whole description family, not by individual sentence.
5. This is an exploratory measurement-validity study. Passing the thresholds below does not prove that the Probe measures human-like thought, absolute concept activation, or a causal mechanism.
6. The new held-out descriptions are author-created. Automated checks catch name leakage and duplicates, but do not replace independent human annotation.

## How to run
1. In Colab, choose **Runtime → Change runtime type → T4 GPU**.
2. Run all cells in order.
3. Add the Colab Secret `GITHUB_RESULTS_TOKEN` (Contents: Read and write for the thesis repository).
4. The feature-extraction cell saves batches locally and uploads grouped checkpoint bundles to GitHub every 512 prompts.
5. The final cell uploads verified results and a ZIP to GitHub, then downloads it locally.
6. For a resumed run, keep the same `RUN_ID`; for a new run, assign a new unique `RUN_ID`.

In [ ]:
%pip -q install "transformers==4.57.6" "accelerate>=1,<2" "huggingface-hub>=0.30,<1" "scikit-learn==1.6.1" pandas numpy matplotlib tqdm requests threadpoolctl tabulate


In [ ]:
import os, re, json, time, math, hashlib, itertools, runpy, zipfile, platform, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
import sklearn
from tqdm.auto import tqdm
from threadpoolctl import threadpool_limits
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.exceptions import ConvergenceWarning

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
LAYERS = [22, 23, 24]
POSITIONS = ["context_end", "assistant_start"]
CONDITIONS = ["neutral", "mention_x", "think_x", "avoid_x", "think_y"]
C = 0.1
BATCH_SIZE = 64
SHUFFLE_ALL_BALANCED_ASSIGNMENTS = True

OUT = Path("/content/qwen_concept_probe_20pairs")
CACHE = OUT / "feature_batches"
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "GPU runtime required. Select a Colab T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "| Transformers:", transformers.__version__, "| sklearn:", sklearn.__version__)


## 0.1. GitHub credentials and run identity

This run saves **verified checkpoint bundles** every 512 prompts, then writes a final result ZIP and short readable reports to the repository. An interrupted Colab session can be resumed by rerunning the notebook **with the same `RUN_ID`**. The notebook fetches completed checkpoint bundles from GitHub before calculating any missing batches.

### One-time setup
- Create a fine-grained Personal Access Token with **Contents: Read and write** only for `ryouy/GraduationThesis`.
- In Colab **Secrets** (key icon), add `GITHUB_RESULTS_TOKEN` and enable notebook access.
- Never paste the token into source code, notebooks, commit messages, or output files.
- Checkpoint and final files are create-only: a changed artifact at the same path causes a hard failure rather than overwriting existing data.

**Repository**: `ryouy/GraduationThesis`  
**Run**: `1110_qwen_probe_20pairs/results/20261110_qwen20pairs_v1/`


In [ ]:
import base64
import io
import requests
from urllib.parse import quote
from google.colab import userdata

GITHUB_REPOSITORY = "ryouy/GraduationThesis"
GITHUB_BRANCH = "main"
EXPERIMENT_SLUG = "1110_qwen_probe_20pairs"
RUN_ID = "20261110_qwen20pairs_v1"  # Keep unchanged to resume this exact experiment. Change for a new independent run.
GITHUB_RUN_PREFIX = f"{EXPERIMENT_SLUG}/results/{RUN_ID}"
REMOTE_CHECKPOINT_INTERVAL = 8  # Eight 64-input feature batches = 512 prompts per GitHub checkpoint.

try:
    GITHUB_TOKEN = userdata.get("GITHUB_RESULTS_TOKEN")
except Exception as exc:
    raise RuntimeError("Add GITHUB_RESULTS_TOKEN in Colab Secrets and enable notebook access before running.") from exc
if not isinstance(GITHUB_TOKEN, str) or not GITHUB_TOKEN.strip():
    raise RuntimeError("Colab Secret GITHUB_RESULTS_TOKEN is missing or empty. A fine-grained token with Contents: read/write on ryouy/GraduationThesis is required.")

GITHUB_HEADERS = {
    "Authorization": "Bearer " + GITHUB_TOKEN.strip(),
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

def _api_url(relative_path):
    return f"https://api.github.com/repos/{GITHUB_REPOSITORY}/contents/{quote(relative_path, safe='/')}"

def _blob_sha(data):
    # GitHub's blob SHA provides an independently checkable content hash.
    import hashlib
    header = f"blob {len(data)}\0".encode("ascii")
    return hashlib.sha1(header + data).hexdigest()

def _remote_info(relative_path):
    response = requests.get(_api_url(relative_path), headers=GITHUB_HEADERS, timeout=60)
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json()

def github_save_bytes(relative_path, data, message):
    """Idempotent create-only write. Existing files must match byte-for-byte or raise."""
    expected = _blob_sha(data)
    current = _remote_info(relative_path)
    if current is not None:
        if current.get("sha") != expected:
            raise RuntimeError(f"GitHub content conflict at {relative_path}; refusing to overwrite a different experiment artifact.")
        print("Already saved (verified):", relative_path, flush=True)
        return
    payload = {
        "message": message,
        "content": base64.b64encode(data).decode("ascii"),
        "branch": GITHUB_BRANCH,
    }
    response = requests.put(_api_url(relative_path), headers=GITHUB_HEADERS, json=payload, timeout=180)
    if response.status_code not in (200, 201):
        raise RuntimeError(f"GitHub upload failed for {relative_path} (HTTP {response.status_code}): {response.text[:300]}")
    returned_sha = response.json()["content"]["sha"]
    assert returned_sha == expected, f"GitHub blob mismatch: {relative_path}"
    print("Saved and verified:", relative_path, flush=True)

def github_download_bytes(relative_path):
    """Retrieve and verify GitHub file bytes; public raw fallback supports >1 MB files."""
    info = _remote_info(relative_path)
    if info is None:
        return None
    raw_url = info.get("download_url")
    assert raw_url, f"No GitHub download URL for {relative_path}"
    response = requests.get(raw_url, timeout=180)
    response.raise_for_status()
    content = response.content
    assert _blob_sha(content) == info["sha"], f"Remote checkpoint hash mismatch: {relative_path}"
    return content

# Check permission before making any expensive GPU calls.
permission = requests.get(f"https://api.github.com/repos/{GITHUB_REPOSITORY}", headers=GITHUB_HEADERS, timeout=60)
permission.raise_for_status()
assert permission.json().get("full_name", "").lower() == GITHUB_REPOSITORY.lower()
print("GitHub destination:", f"https://github.com/{GITHUB_REPOSITORY}/tree/main/{GITHUB_RUN_PREFIX}")
print("Checkpoints are uploaded every", REMOTE_CHECKPOINT_INTERVAL * BATCH_SIZE, "prompts.")


## 1. Reuse the frozen 20-pair training bank

The 20-pair stimulus bank from the earlier Phi-3.5 experiment is reused **only for the training descriptions and pair list**. Its old model outputs are not reused. The new evaluation bank below is separately authored for this Qwen run.

The source file is fetched from the thesis repository and checked against its frozen SHA-256 digest before use.

In [ ]:
import requests

SOURCE_URL = "https://raw.githubusercontent.com/ryouy/GraduationThesis/main/1104_instruction_probe_20pairs/prepare.py"
EXPECTED_PREPARE_SHA256 = "e7b9424f02db0058824aaf742ace6d53ac472dfadad1065ff9ec1f42c33a718b"

source_response = requests.get(SOURCE_URL, timeout=60)
source_response.raise_for_status()
prepare_bytes = source_response.content
actual_sha = hashlib.sha256(prepare_bytes).hexdigest()
assert actual_sha == EXPECTED_PREPARE_SHA256, (
    f"Frozen training source changed: expected {EXPECTED_PREPARE_SHA256}, got {actual_sha}"
)

prepare_path = OUT / "frozen_prepare.py"
prepare_path.write_bytes(prepare_bytes)
source_ns = runpy.run_path(str(prepare_path))

PAIRS = dict(source_ns["PAIRS"])
TRAIN_FACTS = dict(source_ns["FACTS"])
TRAIN_FRAMES = list(source_ns["TRAIN_FRAMES"])

assert len(PAIRS) == 20
assert len(TRAIN_FACTS) == 40
assert all(len(TRAIN_FACTS[c]) == 11 for c in TRAIN_FACTS)
assert len(TRAIN_FRAMES) == 5

print("Frozen training source verified.")
print("Pairs:", len(PAIRS), "| Concepts:", len(TRAIN_FACTS), "| Train families/concept:", 6)
display(pd.DataFrame([{"X": x, "Y": y} for x, y in PAIRS.items()]))


## 2. New held-out descriptions

Each concept has five new semantic description families. The descriptions deliberately shift away from the main features used in the training bank.

For example:
- Elephant training descriptions emphasize trunk, tusks, ears, and body features.
- Elephant held-out descriptions emphasize migration, family groups, calves, low-frequency communication, and habitat modification.

Four wrapper frames are used for each held-out family. The four wrappers are repeated surface forms, not four independent semantic examples.

In [ ]:
HELDOUT = {
    "banana": [
        "The crop is cut while still green and ripened later in temperature-controlled rooms.",
        "The plant's broad leaves often split lengthwise after strong tropical winds.",
        "A plantation worker removed a large hanging flower structure before the next harvest.",
        "The fruit's natural starch gradually changes into sugars during storage.",
        "A smooth puree made from the produce is commonly used as an early infant food."
    ],
    "apple": [
        "Bees visited the orchard during the short period when its trees were in blossom.",
        "Growers grafted a preferred variety onto dwarfing rootstock to control tree height.",
        "The harvest was kept in a low-oxygen storage room so it would remain usable through winter.",
        "A preserve maker relied on the fruit's natural pectin to help a mixture set.",
        "The orchard owner thinned young fruit so the remaining crop would develop more evenly."
    ],
    "elephant": [
        "The matriarch guided several generations along a seasonal route between feeding grounds.",
        "The herd gathered around a newborn calf while the adults remained close together.",
        "Low-frequency rumbles helped distant members of the family group remain in contact.",
        "The animals dug into a dry riverbed to reach water during the drought.",
        "A large herbivore changed the woodland by pushing over small trees and opening clearings."
    ],
    "giraffe": [
        "A newborn was able to stand and walk shortly after being delivered.",
        "Males sometimes compete by swinging their necks against one another.",
        "The animal sleeps for brief periods, often remaining upright.",
        "A mother gave birth while standing, and the newborn dropped to the ground below.",
        "Several individuals browsed among scattered acacia trees across the open savanna."
    ],
    "piano": [
        "The accompanist followed a singer's breathing during a classical art-song recital.",
        "A jazz trio used the instrument to provide the harmonic foundation between bass and drums.",
        "The student practiced separate hand parts before combining them for the examination.",
        "A composer wrote a multi-movement concerto for a solo performer and orchestra.",
        "The stage crew moved the large instrument on a wheeled platform before the concert."
    ],
    "violin": [
        "The concertmaster gave the orchestra its tuning pitch before the conductor entered.",
        "The first chair led the string section through a difficult passage.",
        "The soloist shaped a sustained note with a controlled change in pitch and intensity.",
        "A folk musician played a lively melody during a village dance.",
        "The quartet's upper voice carried the melody while the other players supplied harmony."
    ],
    "cat": [
        "The pet rubbed its cheeks against the furniture to leave a familiar scent.",
        "A nocturnal hunter watched a small movement from the edge of a windowsill.",
        "The animal pressed its front paws alternately into a soft blanket.",
        "Its whiskers helped it judge whether a narrow opening was wide enough to pass through.",
        "The household pet landed upright after jumping down from a high shelf."
    ],
    "dog": [
        "A rescue team followed the animal's scent trail through the rubble after the earthquake.",
        "The companion greeted its owner by spinning excitedly near the doorway.",
        "A shepherd guided a flock across the field without allowing the animals to scatter.",
        "The trained animal detected a hidden substance inside a piece of luggage.",
        "A guide companion helped a pedestrian navigate a busy crossing."
    ],
    "lion": [
        "Females cooperated during a coordinated hunt across the grassland.",
        "The cubs played near a sheltered den while the adults rested nearby.",
        "A deep roar carried across the territory shortly after sunset.",
        "Related females remained together as the core of the social group.",
        "The predator rested in the shade during the hottest part of the day."
    ],
    "tiger": [
        "A solitary hunter dragged its prey into dense cover before feeding.",
        "The animal marked the edge of its range with scent and scratches on tree trunks.",
        "A camera trap recorded one large cat moving alone through a snowy forest.",
        "The predator crossed a river while pursuing prey along the opposite bank.",
        "A conservation team monitored a wide forest territory used by a single individual."
    ],
    "dolphin": [
        "A mother guided her calf through shallow water while it continued to nurse.",
        "Several animals cooperated to herd a school of fish into a compact group.",
        "The marine mammal rested while allowing one half of its brain to remain alert.",
        "A playful animal repeatedly leapt beside the bow of a moving vessel.",
        "Researchers identified individuals by the distinctive patterns of their social calls."
    ],
    "shark": [
        "The predator detected weak electrical signals produced by nearby muscle movements.",
        "A species deposited leathery egg cases that became attached to underwater vegetation.",
        "Sensory structures along the body detected small movements in the surrounding water.",
        "The animal's body coloring made it difficult to distinguish from the water above and below.",
        "A marine predator approached prey by following chemical traces carried through the current."
    ],
    "eagle": [
        "The bird circled above a mountain valley while searching the ground for food.",
        "A breeding pair returned to the same cliff ledge to raise its young.",
        "The raptor detected a small animal from high above an open field.",
        "The nest became larger each breeding season as new branches were added.",
        "The hunter descended rapidly toward prey after watching it from a distant perch."
    ],
    "penguin": [
        "Adults took turns keeping an egg warm during the long breeding period.",
        "A parent recognized its own chick among thousands by listening to its call.",
        "Dense layers of feathers trapped warm air while the bird remained in icy water.",
        "Large breeding colonies gathered on remote shores during the nesting season.",
        "An adult brought food back to the colony and regurgitated it for its young."
    ],
    "butterfly": [
        "The insect migrated over a great distance between seasonal breeding grounds.",
        "A pollinator transferred pollen while visiting several blossoms in the same garden.",
        "The female placed tiny eggs on a particular host plant.",
        "Large groups gathered in a sheltered grove during the cold season.",
        "The species followed a recurring route between its summer and winter habitats."
    ],
    "spider": [
        "The arachnid injected venom to immobilize an insect before feeding.",
        "A female carried a cluster of young on the underside of her body.",
        "Specialized hairs on its legs detected vibrations from nearby movement.",
        "The hunter remained motionless before suddenly ambushing an insect.",
        "As it grew, the animal periodically shed its outer covering."
    ],
    "rose": [
        "A gardener grafted a prized cultivar onto a hardy rootstock.",
        "A florist removed lower leaves before arranging the stems in a vase.",
        "After flowering, the shrub developed a small fruit that remained through autumn.",
        "A perfume maker distilled aromatic oil from the plant's blossoms.",
        "The gardener pruned the woody stems after the first flush of spring growth."
    ],
    "sunflower": [
        "Young plants turned their heads toward the sun during an early stage of growth.",
        "The crop was harvested for cooking oil and livestock feed.",
        "A field of tall annual plants formed a bright border around the farm.",
        "The heavy flower heads began to droop as the seeds matured near the end of summer.",
        "The central disk contained hundreds of tiny individual flowers."
    ],
    "carrot": [
        "The vegetable supplied beta-carotene, which the body can convert into vitamin A.",
        "The produce was preserved in vinegar and spices for use during winter.",
        "A rabbit nibbled the crisp crop from a garden bed.",
        "The plant's finely divided foliage resembled a delicate green fern.",
        "The farmer thinned the seedlings early to give the underground crop enough room to develop."
    ],
    "potato": [
        "Farmers propagated the crop by planting pieces that each contained a viable eye.",
        "The harvest was stored in a cool, dark cellar to delay sprouting.",
        "A field of small pale blossoms appeared above the underground crop.",
        "Starch extracted from the produce was used to thicken a sauce.",
        "The farmer rotated the crop to reduce the build-up of soil-borne disease."
    ],
    "lemon": [
        "The chef added a sharp acidic note to balance a rich sauce.",
        "A preserved slice was packed in salt for use in North African cooking.",
        "The aromatic oil was released when the outer rind was twisted over a dish.",
        "The drink was served with a distinctly tart finish.",
        "The tree produced fragrant white blossoms before the next harvest."
    ],
    "orange": [
        "The outer rind was dried and candied before being used in a confection.",
        "A grove produced fragrant blossoms during the spring flowering season.",
        "The snack was divided into easy-to-share sections for a group of children.",
        "The crop was harvested during the cooler months in a warm growing region.",
        "The fruit was packed into crates after being sorted by size and ripeness."
    ],
    "bread": [
        "The baker scored the dough before placing it inside a hot stone chamber.",
        "A starter culture fermented on the counter for several days before being used.",
        "The crust crackled as steam escaped from the loaf while it cooled.",
        "The dough expanded after resting in a warm place.",
        "A picnic lunch was prepared by placing several fillings between two slices."
    ],
    "rice": [
        "The crop thrived in flooded fields during the warm monsoon season.",
        "The grains were milled to remove their outer husks before cooking.",
        "A sticky variety was shaped into small cakes for a seasonal festival.",
        "The harvest was dried before being stored in large sacks.",
        "The plant grew in long clusters above shallow standing water."
    ],
    "bicycle": [
        "The rider changed gears before climbing a steep hill.",
        "A racing group traveled in a tightly organized peloton.",
        "The frame was secured to a rack outside the railway station.",
        "The athlete wore clipped shoes during a long-distance road event.",
        "A mechanic adjusted the spokes to make a wheel run straight."
    ],
    "motorcycle": [
        "The rider shifted gears with a foot lever while operating a hand clutch.",
        "The exhaust produced a low rumble while the vehicle waited at a traffic light.",
        "A touring rider packed luggage beside the rear seat before a long trip.",
        "The machine leaned sharply into a curve on a mountain road.",
        "A mechanic checked tire pressure and the tension of the drive chain."
    ],
    "train": [
        "A conductor checked tickets as passengers boarded at the platform.",
        "The locomotive hauled freight across the continent overnight.",
        "The timetable listed a departure from the central terminal.",
        "A high-speed service connected two major cities in less than an hour.",
        "A signal changed to green before the vehicle continued along its route."
    ],
    "airplane": [
        "Cabin crew demonstrated the emergency exits before departure.",
        "Passengers fastened their seat belts as the aircraft climbed.",
        "The pilot requested clearance from air traffic control.",
        "A baggage carousel delivered the travelers' luggage after arrival.",
        "The vehicle crossed an ocean in several hours."
    ],
    "boat": [
        "The crew dropped anchor in a sheltered bay and remained there overnight.",
        "A harbor pilot guided the vessel through a narrow channel.",
        "The passengers put on life jackets before leaving the dock.",
        "The craft rocked gently as waves passed beneath it.",
        "A sail caught the wind during a coastal voyage."
    ],
    "submarine": [
        "The crew remained below the surface for weeks without coming ashore.",
        "Hydrophones detected distant propeller noise in the dark water.",
        "The vessel launched a torpedo during a naval exercise.",
        "A sonar operator mapped the seabed in near darkness.",
        "The crew used an escape chamber during a rescue drill."
    ],
    "hammer": [
        "A blacksmith struck an anvil to flatten a heated bar of metal.",
        "Construction workers demolished a brick wall using repeated impacts.",
        "A courtroom official used a wooden mallet to call the session to order.",
        "The carpenter drove a wooden wedge into a tight joint.",
        "A metalworker shaped a hot piece of material through repeated blows."
    ],
    "screwdriver": [
        "An electrician tightened a terminal screw inside a wall switch.",
        "A repairer opened a small electronic casing to replace a battery.",
        "A technician adjusted a precision fastener on a camera tripod.",
        "The handle rotated while the metal tip remained seated in a narrow recess.",
        "A small tool was magnetized so that tiny fasteners would stay attached."
    ],
    "scissors": [
        "A tailor trimmed excess fabric around a paper pattern.",
        "A barber shaped a client's hair with repeated small snips.",
        "A child made paper snowflakes by folding and cutting a sheet.",
        "A gardener removed a dead flower head with a bypass cutting tool.",
        "The implement opened and closed repeatedly using one hand."
    ],
    "knife": [
        "A chef filleted a fish along its backbone.",
        "The cook sharpened the edge against a whetstone before preparing dinner.",
        "A surgeon made a precise incision during an operation.",
        "The butcher carved thin portions from a roasted joint of meat.",
        "A hiker prepared kindling with a single-edged cutting implement."
    ],
    "clock": [
        "The alarm rang at dawn after being set the previous evening.",
        "A pendulum swung steadily inside a wooden case.",
        "The station announced that the mechanism had lost three minutes.",
        "The tower chimed at the beginning of every hour.",
        "A watchmaker replaced a tiny spring inside the timekeeping mechanism."
    ],
    "calendar": [
        "The office marked public holidays and project deadlines on a yearly planner.",
        "A team arranged its milestones across the twelve months of the year.",
        "The appointment was moved from one weekday to another.",
        "The wall display showed school terms and vacation periods.",
        "The organizer crossed off each day until the wedding ceremony."
    ],
    "refrigerator": [
        "The freezer compartment formed a layer of ice around the trays.",
        "A sensor alerted the owner when the door remained open too long.",
        "The compressor cycled on and off to maintain a stable internal temperature.",
        "Leftovers were stored safely overnight instead of being left on the counter.",
        "The kitchen appliance dispensed chilled water through a small opening in its door."
    ],
    "oven": [
        "A baker preheated the chamber before placing a tray of cookies inside.",
        "The roast developed a browned crust under sustained dry heat.",
        "A thermostat controlled the temperature during a long baking process.",
        "A fan circulated hot air around the food during cooking.",
        "The cook browned vegetables under an intense upper heating element."
    ],
    "camera": [
        "A photographer adjusted exposure before capturing a fast-moving scene.",
        "A memory card stored hundreds of digital image files.",
        "A flash illuminated the subject in a dimly lit room.",
        "The device recorded a short video during the ceremony.",
        "A portrait was carefully framed while looking through a viewfinder."
    ],
    "telescope": [
        "An astronomer tracked a comet as it moved across the night sky.",
        "A large mirror collected faint light from a distant nebula.",
        "The observatory opened its dome after sunset.",
        "The instrument resolved two stars that appeared close together.",
        "A motorized mount compensated for the rotation of the Earth."
    ]
}

assert set(HELDOUT) == set(TRAIN_FACTS)
assert all(len(v) == 5 for v in HELDOUT.values())

FEATURE_AUDIT = {
    "banana/apple": "Training: peel, color, shape, taste, common food uses. Held-out: crop handling, plant growth, storage, pectin, orchard pollination and cultivar management.",
    "elephant/giraffe": "Training: trunk, tusks, ears, neck, spots and body shape. Held-out: migration, family groups, calves, vocal communication, birth and savanna behavior.",
    "piano/violin": "Training: keyboard, keys, strings, hammers and bow. Held-out: accompaniment, ensemble roles, repertoire, rehearsal and performance contexts.",
    "cat/dog": "Training: purring, claws, barking, fetching and leash walking. Held-out: scent work, territory marking, service tasks, body language and household behavior.",
    "lion/tiger": "Training: mane, stripes, prides and solitary appearance. Held-out: cooperative hunting, cub care, territory, habitat use and camera-trap observations.",
    "dolphin/shark": "Training: blowhole, echolocation, teeth, gills and body shape. Held-out: maternal care, cooperative feeding, sleep, electroreception and reproduction.",
    "eagle/penguin": "Training: flight, talons, hooked beak, flippers and flightlessness. Held-out: nesting, incubation, chick recognition, colony behavior and hunting observation.",
    "butterfly/spider": "Training: wings, caterpillar, chrysalis, silk and webs. Held-out: migration, pollination, egg placement, vibration sensing and predatory behavior.",
    "rose/sunflower": "Training: petals, fragrance, prickles, yellow flower heads and seeds. Held-out: grafting, pruning, perfume production, growth orientation and oil production.",
    "carrot/potato": "Training: orange roots, leafy tops, tubers, eyes and common cooking forms. Held-out: nutrition, preservation, crop propagation, storage and plant growth.",
    "lemon/orange": "Training: sourness, color, peel, segments and juice. Held-out: preservation, rind oils, blossoms, harvest and culinary balancing.",
    "bread/rice": "Training: flour dough, loaves, yeast, cereal grains, paddies and sushi. Held-out: baking process, fermentation, crust, milling, drying and crop cultivation.",
    "bicycle/motorcycle": "Training: pedals, chain, human power, engine and throttle. Held-out: racing, gears, clutch operation, touring, road handling and maintenance.",
    "train/airplane": "Training: rails, carriages, stations, fixed wings and runway. Held-out: ticket inspection, freight, timetables, cabin safety and air-traffic control.",
    "boat/submarine": "Training: surface travel, oars, hull, underwater travel, ballast and periscope. Held-out: anchoring, harbor navigation, life jackets, hydrophones and sonar.",
    "hammer/screwdriver": "Training: impact, head, nails, claw, fasteners and tip. Held-out: blacksmithing, demolition, courtroom mallet use, electronics repair and precision adjustment.",
    "scissors/knife": "Training: paired blades, finger loops, single blade and food cutting. Held-out: tailoring, hair cutting, paper craft, filleting, surgery and carving.",
    "clock/calendar": "Training: hands, ticking, dial, hours, dates and month grids. Held-out: alarms, pendulums, mechanical repair, project planning and appointments.",
    "refrigerator/oven": "Training: cold storage, compressor, shelves, baking chamber and roasting. Held-out: freezer behavior, door sensors, leftovers, preheating, convection and broiling.",
    "camera/telescope": "Training: image sensor, shutter, lens, photographs, stars and magnification. Held-out: exposure, memory cards, flash, viewfinder, comet tracking and observatory operation."
}

display(pd.DataFrame([{"Pair": k, "Feature separation (manual audit)": v} for k, v in FEATURE_AUDIT.items()]))


## 3. Construct training and evaluation prompts

The Probe is trained on the first six training-description families per concept. In each of six folds, one complete family from X and one from Y are held out from training, leaving 25 training examples per class.

Evaluation uses five newly authored semantic families per concept, each expressed through four wrapper frames. Every evaluation family is measured under all five instruction conditions.

In [ ]:
EVAL_FRAMES = [
    "A field report notes: {s}",
    "An observer recorded: {s}",
    "The account states: {s}",
    "A research note describes the scene: {s}"
]

INSTRUCTIONS = {
    "neutral": "Read the context carefully.",
    "mention_x": "The topic word is {x}.",
    "think_x": "Think about {x}.",
    "avoid_x": "Do not think about {x}.",
    "think_y": "Think about {y}."
}

def make_prompt(x, y, condition, text):
    instruction = INSTRUCTIONS[condition].format(x=x, y=y)
    prefix = f"Instruction: {instruction}\n\nContext:\n"
    prompt = prefix + text + "\n\nGive one short response about the context."
    context_end_char = len(prefix) + len(text) - 1
    return prompt, context_end_char

training_rows = []
for x, y in PAIRS.items():
    for concept, label in [(x, 1), (y, 0)]:
        for family_idx, fact in enumerate(TRAIN_FACTS[concept][:6]):
            for wrapper_idx, frame in enumerate(TRAIN_FRAMES):
                text = frame.format(s=fact)
                prompt, end_char = make_prompt(x, y, "neutral", text)
                training_rows.append({
                    "pair": x, "comparison": y, "concept": concept, "label": label,
                    "cue_kind": "x" if label == 1 else "y",
                    "family_index": family_idx,
                    "family": f"{concept}_train_{family_idx}",
                    "wrapper_index": wrapper_idx,
                    "text": text, "condition": "neutral",
                    "prompt": prompt, "context_end_char": end_char
                })

evaluation_rows = []
for x, y in PAIRS.items():
    for concept, cue_kind in [(x, "x"), (y, "y")]:
        for family_idx, sentence in enumerate(HELDOUT[concept]):
            for wrapper_idx, frame in enumerate(EVAL_FRAMES):
                text = frame.format(s=sentence)
                for condition in CONDITIONS:
                    prompt, end_char = make_prompt(x, y, condition, text)
                    evaluation_rows.append({
                        "pair": x, "comparison": y, "concept": concept,
                        "label": 1 if cue_kind == "x" else 0,
                        "cue_kind": cue_kind,
                        "family_index": family_idx,
                        "family": f"{concept}_eval_{family_idx}",
                        "wrapper_index": wrapper_idx,
                        "text": text, "condition": condition,
                        "prompt": prompt, "context_end_char": end_char
                    })

train_df = pd.DataFrame(training_rows)
eval_df = pd.DataFrame(evaluation_rows)

assert len(train_df) == 40 * 6 * 5 == 1200
assert len(eval_df) == 20 * 2 * 5 * 4 * 5 == 4000
assert train_df.groupby("concept").size().eq(30).all()
assert eval_df.groupby(["pair", "condition", "cue_kind"]).size().eq(20).all()

# Name-leakage check: neither X nor Y may appear as a standalone word in held-out text.
def has_concept_name(text, concept):
    # Exact word plus common English plural suffixes.
    pattern = r"(?i)(?<![A-Za-z])" + re.escape(concept) + r"(?:s|es)?(?![A-Za-z])"
    return bool(re.search(pattern, text))

leaks = []
for row in eval_df.drop_duplicates(["pair", "family", "text"]).itertuples():
    if has_concept_name(row.text, row.concept) or has_concept_name(row.text, row.comparison):
        leaks.append((row.pair, row.concept, row.text))
assert not leaks, f"Concept-name leakage found: {leaks[:5]}"

# Exact duplicate checks across held-out families and between train/evaluation texts.
def norm_text(s):
    return re.sub(r"\s+", " ", s.strip().lower())

heldout_unique = eval_df.drop_duplicates(["concept", "family"])[["concept", "family", "text"]]
assert heldout_unique.groupby("concept").size().eq(5).all()
assert heldout_unique["text"].map(norm_text).nunique() == len(heldout_unique)
train_texts = set(train_df["text"].map(norm_text))
heldout_texts = set(heldout_unique["text"].map(norm_text))
assert not (train_texts & heldout_texts), "Exact training/evaluation text overlap found."

# No duplicated prompts within the same design.
assert not train_df["prompt"].duplicated().any()
assert not eval_df["prompt"].duplicated().any()

PROMPTS = list(dict.fromkeys(train_df["prompt"].tolist() + eval_df["prompt"].tolist()))
PROMPT_TO_INDEX = {p: i for i, p in enumerate(PROMPTS)}
PROMPT_HASH = hashlib.sha256("\n".join(PROMPTS).encode("utf-8")).hexdigest()

print("Training rows:", len(train_df))
print("Evaluation rows:", len(eval_df))
print("Unique forward prompts:", len(PROMPTS))
print("Held-out semantic families:", len(heldout_unique))
print("Concept-name leakage: none")
print("Exact train/evaluation text overlap: none")

train_df.to_csv(OUT / "training_design.csv", index=False)
eval_df.to_csv(OUT / "evaluation_design.csv", index=False)
heldout_unique.to_csv(OUT / "heldout_families.csv", index=False)
(OUT / "feature_audit.json").write_text(json.dumps(FEATURE_AUDIT, ensure_ascii=False, indent=2), encoding="utf-8")


## 4. Load the pinned Qwen model

The model is loaded in FP32 with eager attention to keep the extraction setup explicit and reproducible. No text generation is performed; only prompt hidden states are extracted.

In [ ]:
assert torch.cuda.is_available()
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, use_fast=True, trust_remote_code=False
)
assert tokenizer.is_fast

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.float32,
    attn_implementation="eager",
    low_cpu_mem_usage=True,
    trust_remote_code=False
).cuda().eval()

assert model.config.num_hidden_layers == 24
assert model.config.hidden_size == 896
assert str(next(model.parameters()).dtype) == "torch.float32"

print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Layers:", model.config.num_hidden_layers)
print("Hidden size:", model.config.hidden_size)
print("Attention:", getattr(model.config, "_attn_implementation", "unknown"))


## 5. Extract hidden states with resumable batches

For each prompt, extract:
- `context_end`: the hidden state at the final token of the context sentence.
- `assistant_start`: the final prompt token immediately before the model begins its response.

Each position is evaluated with separate Probes. Absolute scores from different positions are not treated as directly interchangeable.

In [ ]:
FEATURE_KEYS = [f"L{layer}_{position}" for layer in LAYERS for position in POSITIONS]
N_PROMPTS = len(PROMPTS)
HIDDEN_SIZE = model.config.hidden_size

def get_prompt_features(prompt, context_end_char):
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )
    prompt_start = rendered.index(prompt)
    target_char = prompt_start + context_end_char

    encoded = tokenizer(
        rendered,
        return_tensors="pt",
        return_offsets_mapping=True,
        add_special_tokens=False
    )
    offsets = encoded.pop("offset_mapping")[0].tolist()
    hits = [i for i, (lo, hi) in enumerate(offsets) if lo <= target_char < hi]
    assert len(hits) == 1, f"Could not map context-end character: {prompt[:100]}"
    positions = {
        "context_end": hits[0],
        "assistant_start": encoded["input_ids"].shape[1] - 1
    }

    model_inputs = {k: v.cuda() for k, v in encoded.items()}
    with torch.inference_mode():
        output = model(
            **model_inputs,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True
        )

    result = {}
    for layer in LAYERS:
        for position in POSITIONS:
            key = f"L{layer}_{position}"
            result[key] = output.hidden_states[layer][0, positions[position]].float().cpu().numpy()
    del output
    return result

# Prompt metadata is fixed before extraction.
prompt_meta = {}
for row in train_df.itertuples():
    prompt_meta[row.prompt] = row.context_end_char
for row in eval_df.itertuples():
    prompt_meta[row.prompt] = row.context_end_char
assert len(prompt_meta) == N_PROMPTS

# Resumable batch extraction. Completed batches are skipped on rerun.
batch_manifest = {
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "prompt_hash": PROMPT_HASH,
    "n_prompts": N_PROMPTS,
    "layers": LAYERS,
    "positions": POSITIONS,
    "dtype": "float32",
    "training_source_sha256": EXPECTED_PREPARE_SHA256,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "github_run_prefix": GITHUB_RUN_PREFIX
}
(OUT / "extraction_config.json").write_text(json.dumps(batch_manifest, indent=2), encoding="utf-8")


def _checkpoint_remote_path(group):
    return f"{GITHUB_RUN_PREFIX}/checkpoint_chunks/chunk_{group:03d}.zip"

def _bundle_info(group):
    first_batch = group * REMOTE_CHECKPOINT_INTERVAL
    n_total_batches = math.ceil(N_PROMPTS / BATCH_SIZE)
    last_batch = min((group + 1) * REMOTE_CHECKPOINT_INTERVAL, n_total_batches)
    return range(first_batch, last_batch)

def restore_remote_checkpoint_bundles():
    n_groups = math.ceil(math.ceil(N_PROMPTS / BATCH_SIZE) / REMOTE_CHECKPOINT_INTERVAL)
    restored = 0
    for group in range(n_groups):
        batch_ids = _bundle_info(group)
        expected_files = [f"batch_{b:04d}.npz" for b in batch_ids]
        if all((CACHE / name).exists() for name in expected_files):
            continue
        content = github_download_bytes(_checkpoint_remote_path(group))
        if content is None:
            continue
        with zipfile.ZipFile(io.BytesIO(content)) as zf:
            assert sorted(zf.namelist()) == sorted(expected_files + ["checkpoint_manifest.json"])
            remote_manifest = json.loads(zf.read("checkpoint_manifest.json"))
            assert remote_manifest == batch_manifest, "Remote checkpoint configuration mismatch; do not reuse this run ID."
            for name in expected_files:
                data = zf.read(name)
                # Check each batch's prompt/config identity BEFORE using cached bytes.
                with np.load(io.BytesIO(data)) as z:
                    assert str(z["prompt_hash"].item()) == PROMPT_HASH
                    for key in FEATURE_KEYS:
                        assert z[key].shape[1] == HIDDEN_SIZE
                path = CACHE / name
                if path.exists():
                    assert path.read_bytes() == data, "Local/remote batch conflict"
                else:
                    path.write_bytes(data)
                    restored += 1
    print(f"Restored {restored} batch files from GitHub checkpoints.", flush=True)

def upload_group_when_complete(group):
    names = [f"batch_{b:04d}.npz" for b in _bundle_info(group)]
    paths = [CACHE / name for name in names]
    if not all(p.exists() for p in paths):
        return
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("checkpoint_manifest.json", json.dumps(batch_manifest, sort_keys=True))
        for p in paths:
            archive.write(p, arcname=p.name)
    github_save_bytes(
        _checkpoint_remote_path(group), buf.getvalue(),
        f"Save Qwen Probe 20-pair feature checkpoint group {group:03d}"
    )

restore_remote_checkpoint_bundles()

for batch_no, start in enumerate(tqdm(range(0, N_PROMPTS, BATCH_SIZE), desc="Feature batches")):
    stop = min(start + BATCH_SIZE, N_PROMPTS)
    batch_path = CACHE / f"batch_{batch_no:04d}.npz"
    if batch_path.exists():
        # Reuse only a batch generated for the same ordered prompt set.
        with np.load(batch_path) as z:
            assert str(z["prompt_hash"].item()) == PROMPT_HASH
            assert int(z["start"].item()) == start and int(z["stop"].item()) == stop
        continue

    bucket = {key: [] for key in FEATURE_KEYS}
    for idx in range(start, stop):
        prompt = PROMPTS[idx]
        feats = get_prompt_features(prompt, prompt_meta[prompt])
        for key in FEATURE_KEYS:
            bucket[key].append(feats[key].astype(np.float16))

    save_dict = {key: np.stack(values) for key, values in bucket.items()}
    save_dict["prompt_hash"] = np.array(PROMPT_HASH)
    save_dict["start"] = np.array(start)
    save_dict["stop"] = np.array(stop)
    np.savez_compressed(batch_path, **save_dict)
    # Persist remotely every 8 batches; interruption loses at most the current interval.
    if (batch_no + 1) % REMOTE_CHECKPOINT_INTERVAL == 0 or stop == N_PROMPTS:
        upload_group_when_complete(batch_no // REMOTE_CHECKPOINT_INTERVAL)

# Also upload any completed local groups after a previous interrupted upload.
for group in range(math.ceil(math.ceil(N_PROMPTS / BATCH_SIZE) / REMOTE_CHECKPOINT_INTERVAL)):
    upload_group_when_complete(group)

print("Feature extraction batches:", len(list(CACHE.glob("batch_*.npz"))))


In [ ]:
# Assemble all cached batches into one indexed feature matrix per layer/position.
FEATURES = {
    key: np.empty((N_PROMPTS, HIDDEN_SIZE), dtype=np.float32)
    for key in FEATURE_KEYS
}

for batch_no, start in enumerate(range(0, N_PROMPTS, BATCH_SIZE)):
    stop = min(start + BATCH_SIZE, N_PROMPTS)
    batch_path = CACHE / f"batch_{batch_no:04d}.npz"
    assert batch_path.exists(), f"Missing feature batch: {batch_path}"
    with np.load(batch_path) as z:
        assert str(z["prompt_hash"].item()) == PROMPT_HASH
        for key in FEATURE_KEYS:
            FEATURES[key][start:stop] = z[key].astype(np.float32)

assert all(np.isfinite(v).all() for v in FEATURES.values())
PROMPT_TO_INDEX = {p: i for i, p in enumerate(PROMPTS)}
print("Feature matrices assembled:", {k: v.shape for k, v in FEATURES.items()})


## 6. Fit Probes, test family robustness, and run exact label-shuffle controls

For each pair:
- Six training folds leave out one complete family from X and one from Y.
- The remaining 10 training families contain 5 families from each class, with five wrapper examples per family.
- The family-level null control enumerates all \(\binom{10}{5}=252\) balanced assignments.
- The primary validation score is AUC over the 10 held-out evaluation-family means.
- A second diagnostic removes each matched X/Y evaluation-family pair in turn and recalculates AUC.

The full shuffled AUC distribution is retained, including its minimum and maximum.

In [ ]:
def make_probe():
    return LogisticRegression(
        C=C,
        solver="liblinear",
        max_iter=2000,
        random_state=SEED
    )

def family_auc(frame):
    family_scores = (
        frame.groupby(["family", "cue_kind"], as_index=False)["score"]
        .mean()
    )
    y_true = (family_scores["cue_kind"] == "x").astype(int).to_numpy()
    return float(roc_auc_score(y_true, family_scores["score"].to_numpy()))

def auc_after_family_drop(frame, drop_idx):
    kept = frame[frame["family_index"] != drop_idx]
    return family_auc(kept)

fold_rows = []
family_drop_rows = []
shuffle_rows = []
probe_meta_rows = []

warnings.simplefilter("error", ConvergenceWarning)

with threadpool_limits(limits=2):
    for x, y in tqdm(PAIRS.items(), desc="Concept pairs"):
        train_pair = train_df[train_df["concept"].isin([x, y])].copy()
        eval_pair = eval_df[eval_df["pair"] == x].copy()

        assert train_pair.groupby("concept").size().to_dict() == {x: 30, y: 30}
        assert eval_pair.groupby(["condition", "cue_kind"]).size().eq(20).all()

        for fold in range(6):
            tr = train_pair[train_pair["family_index"] != fold].copy()
            assert tr.groupby("concept").size().to_dict() == {x: 25, y: 25}
            assert tr["family"].nunique() == 10

            train_idx = np.array([PROMPT_TO_INDEX[p] for p in tr["prompt"]])
            y_true_train = tr["label"].to_numpy(dtype=int)
            family_ids = sorted(tr["family"].unique())
            assert len(family_ids) == 10
            assert sum(tr.drop_duplicates("family")["label"] == 1) == 5

            for layer in LAYERS:
                for position in POSITIONS:
                    key = f"L{layer}_{position}"
                    X_train = FEATURES[key][train_idx]
                    scaler = StandardScaler().fit(X_train)
                    X_train_z = scaler.transform(X_train)

                    clf = make_probe().fit(X_train_z, y_true_train)
                    probe_meta_rows.append({
                        "pair": x, "comparison": y, "fold": fold,
                        "layer": layer, "position": position,
                        "n_train": len(tr), "n_train_per_class": 25,
                        "n_train_families": 10
                    })

                    for condition in CONDITIONS:
                        ev = eval_pair[eval_pair["condition"] == condition].copy()
                        ev_idx = np.array([PROMPT_TO_INDEX[p] for p in ev["prompt"]])
                        X_eval = FEATURES[key][ev_idx]
                        scores = clf.decision_function(scaler.transform(X_eval))
                        ev["score"] = scores

                        auc_fam = family_auc(ev)
                        auc_rows = float(roc_auc_score(ev["label"], scores))
                        fold_rows.append({
                            "pair": x, "comparison": y, "fold": fold,
                            "layer": layer, "position": position,
                            "condition": condition,
                            "auc_family_macro": auc_fam,
                            "auc_wrapper_rows_descriptive": auc_rows,
                            "n_semantic_families": 10,
                            "n_wrapper_rows": len(ev)
                        })

                        for drop_idx in range(5):
                            family_drop_rows.append({
                                "pair": x, "comparison": y, "fold": fold,
                                "layer": layer, "position": position,
                                "condition": condition,
                                "dropped_eval_family_index": drop_idx,
                                "auc_family_macro": auc_after_family_drop(ev, drop_idx)
                            })

                    # Exact family-level permutation control:
                    # all 252 balanced 5-vs-5 label assignments among the 10 train families.
                    if layer == 24 and position == "assistant_start":
                        neutral = eval_pair[eval_pair["condition"] == "neutral"].copy()
                        neutral_idx = np.array([PROMPT_TO_INDEX[p] for p in neutral["prompt"]])
                        X_neutral = FEATURES[key][neutral_idx]
                        X_neutral_z = scaler.transform(X_neutral)

                        actual_neutral_scores = clf.decision_function(X_neutral_z)
                        actual_neutral = neutral.copy()
                        actual_neutral["score"] = actual_neutral_scores
                        actual_auc = family_auc(actual_neutral)

                        assignments = list(itertools.combinations(family_ids, 5))
                        assert len(assignments) == 252

                        for perm_id, positive_families in enumerate(assignments):
                            positive_families = set(positive_families)
                            perm_labels = np.array([
                                1 if family in positive_families else 0
                                for family in tr["family"]
                            ], dtype=int)
                            assert perm_labels.sum() == 25

                            null_clf = make_probe().fit(X_train_z, perm_labels)
                            null_scores = null_clf.decision_function(X_neutral_z)
                            null_frame = neutral.copy()
                            null_frame["score"] = null_scores
                            null_auc = family_auc(null_frame)

                            is_true = (
                                all((family in positive_families) == (tr[tr["family"] == family]["label"].iloc[0] == 1)
                                    for family in family_ids)
                            )
                            shuffle_rows.append({
                                "pair": x, "comparison": y, "fold": fold,
                                "perm_id": perm_id,
                                "is_true_assignment": bool(is_true),
                                "auc_family_macro": null_auc
                            })

                        print(
                            f"{x}/{y} fold {fold}: true neutral AUC={actual_auc:.3f}; "
                            f"252 family-level permutations completed",
                            flush=True
                        )

fold_auc_df = pd.DataFrame(fold_rows)
family_drop_df = pd.DataFrame(family_drop_rows)
shuffle_df = pd.DataFrame(shuffle_rows)
probe_meta_df = pd.DataFrame(probe_meta_rows)

assert len(shuffle_df) == 20 * 6 * 252
assert shuffle_df.groupby(["pair", "fold"]).size().eq(252).all()
assert shuffle_df.groupby(["pair", "fold"])["is_true_assignment"].sum().eq(1).all()

fold_auc_df.to_csv(OUT / "fold_auc.csv", index=False)
family_drop_df.to_csv(OUT / "family_drop_auc.csv", index=False)
shuffle_df.to_csv(OUT / "shuffle_null_full.csv", index=False)
probe_meta_df.to_csv(OUT / "probe_metadata.csv", index=False)

print("True Probe evaluations:", len(fold_auc_df))
print("Family-drop evaluations:", len(family_drop_df))
print("Exact shuffled fits:", len(shuffle_df))


## 7. Summaries and exploratory stability flags

Exploratory thresholds:
- Neutral family-macro AUC of at least 0.80 is a strong descriptive result.
- AUC of at least 0.75 is used as a softer stability reference.
- A permutation proportion of at most 0.05 means the observed AUC is in the upper 5% of the enumerated family-shuffle distribution.

A pair is flagged as an **exploratory stable candidate** only if:
- at least 5/6 training folds have neutral AUC ≥ 0.75,
- at least 5/6 folds have family-shuffle proportion ≤ 0.05,
- the median worst-case family-drop AUC is ≥ 0.70.

These are screening rules, not confirmatory statistical tests. No multiple-comparison correction is applied across the 20 pairs.

In [ ]:
primary = fold_auc_df[
    (fold_auc_df["layer"] == 24) &
    (fold_auc_df["position"] == "assistant_start") &
    (fold_auc_df["condition"] == "neutral")
].copy()

null_summary_rows = []
for (pair, fold), g in shuffle_df.groupby(["pair", "fold"]):
    true_row = primary[(primary["pair"] == pair) & (primary["fold"] == fold)]
    assert len(true_row) == 1
    actual = float(true_row["auc_family_macro"].iloc[0])
    vals = g["auc_family_macro"].to_numpy(dtype=float)
    null_summary_rows.append({
        "pair": pair,
        "fold": fold,
        "actual_auc": actual,
        "shuffle_n": len(vals),
        "shuffle_min": float(np.min(vals)),
        "shuffle_q05": float(np.quantile(vals, 0.05)),
        "shuffle_median": float(np.median(vals)),
        "shuffle_q95": float(np.quantile(vals, 0.95)),
        "shuffle_max": float(np.max(vals)),
        "shuffle_mean": float(np.mean(vals)),
        "shuffle_sd": float(np.std(vals, ddof=1)),
        "proportion_shuffle_auc_ge_actual": float(np.mean(vals >= actual - 1e-12)),
        "exact_upper_tail_proportion": float(np.sum(vals >= actual - 1e-12) / len(vals))
    })

null_summary_df = pd.DataFrame(null_summary_rows)
null_summary_df.to_csv(OUT / "shuffle_summary.csv", index=False)

family_drop_primary = family_drop_df[
    (family_drop_df["layer"] == 24) &
    (family_drop_df["position"] == "assistant_start") &
    (family_drop_df["condition"] == "neutral")
].copy()

family_drop_min = (
    family_drop_primary.groupby(["pair", "fold"], as_index=False)
    .agg(worst_family_drop_auc=("auc_family_macro", "min"),
         median_family_drop_auc=("auc_family_macro", "median"))
)

pair_summary_rows = []
for pair, g in primary.groupby("pair"):
    ns = null_summary_df[null_summary_df["pair"] == pair]
    fd = family_drop_min[family_drop_min["pair"] == pair]
    aucs = g["auc_family_macro"].to_numpy()
    pvals = ns["exact_upper_tail_proportion"].to_numpy()
    worst = fd["worst_family_drop_auc"].to_numpy()

    auc_pass = int(np.sum(aucs >= 0.75))
    perm_pass = int(np.sum(pvals <= 0.05))
    family_robust = float(np.median(worst))

    pair_summary_rows.append({
        "pair": pair,
        "comparison": PAIRS[pair],
        "folds": len(aucs),
        "mean_neutral_auc": float(np.mean(aucs)),
        "median_neutral_auc": float(np.median(aucs)),
        "min_neutral_auc": float(np.min(aucs)),
        "max_neutral_auc": float(np.max(aucs)),
        "folds_auc_ge_075": auc_pass,
        "folds_auc_ge_080": int(np.sum(aucs >= 0.80)),
        "median_shuffle_upper_tail_proportion": float(np.median(pvals)),
        "folds_shuffle_p_le_005": perm_pass,
        "median_worst_family_drop_auc": family_robust,
        "exploratory_stable_candidate": bool(
            auc_pass >= 5 and perm_pass >= 5 and family_robust >= 0.70
        )
    })

pair_summary_df = pd.DataFrame(pair_summary_rows).sort_values("pair")
pair_summary_df.to_csv(OUT / "pair_summary.csv", index=False)

condition_summary_df = (
    fold_auc_df.groupby(["layer", "position", "condition"], as_index=False)
    .agg(
        mean_family_auc=("auc_family_macro", "mean"),
        median_family_auc=("auc_family_macro", "median"),
        proportion_auc_ge_075=("auc_family_macro", lambda s: float(np.mean(s >= 0.75))),
        proportion_auc_ge_080=("auc_family_macro", lambda s: float(np.mean(s >= 0.80))),
        n_pair_fold_fits=("auc_family_macro", "size")
    )
)
condition_summary_df.to_csv(OUT / "condition_layer_position_summary.csv", index=False)

display(pair_summary_df)
display(condition_summary_df)


In [ ]:
# Visualize the main validation result by pair and condition.
plot_df = fold_auc_df[
    (fold_auc_df["layer"] == 24) &
    (fold_auc_df["position"] == "assistant_start")
].copy()

condition_order = CONDITIONS
pairs_order = list(PAIRS.keys())
fig, axes = plt.subplots(5, 4, figsize=(17, 18), sharey=True, layout="constrained")
for ax, pair in zip(axes.flat, pairs_order):
    sub = plot_df[plot_df["pair"] == pair]
    for cond in condition_order:
        vals = sub[sub["condition"] == cond].sort_values("fold")["auc_family_macro"].to_numpy()
        ax.plot(range(len(vals)), vals, marker="o", linewidth=1.2, label=cond)
    ax.axhline(0.5, linestyle="--", linewidth=0.8)
    ax.axhline(0.75, linestyle=":", linewidth=0.8)
    ax.set_title(f"{pair} / {PAIRS[pair]}", fontsize=9)
    ax.set_xticks(range(6))
    ax.set_xlabel("Training family omitted", fontsize=7)
    ax.set_ylim(0, 1.02)
    ax.tick_params(labelsize=7)
axes.flat[0].legend(fontsize=7, frameon=False)
fig.suptitle("Layer 24 · assistant-start · family-macro AUC (six overlapping training folds)")
fig.savefig(OUT / "condition_auc_by_pair.png", dpi=160)
plt.show()
plt.close(fig)

# Full permutation distributions for the primary neutral condition.
fig, axes = plt.subplots(5, 4, figsize=(17, 18), sharex=True, sharey=True, layout="constrained")
for ax, pair in zip(axes.flat, pairs_order):
    sub = shuffle_df[(shuffle_df["pair"] == pair) & (shuffle_df["fold"] == 0)]
    actual = float(primary[(primary["pair"] == pair) & (primary["fold"] == 0)]["auc_family_macro"].iloc[0])
    ax.hist(sub["auc_family_macro"], bins=np.linspace(0, 1, 21), density=False)
    ax.axvline(actual, linestyle="--", linewidth=1.4)
    ax.set_title(f"{pair} / observed={actual:.2f}", fontsize=9)
    ax.tick_params(labelsize=7)
fig.suptitle("Exact family-level shuffle AUC distributions (fold 0; 252 assignments per pair)")
fig.savefig(OUT / "shuffle_distributions_fold0.png", dpi=160)
plt.show()
plt.close(fig)

# Family-swap robustness: remove one matched X/Y evaluation family at a time.
fig, ax = plt.subplots(figsize=(13, 5))
fd_plot = family_drop_primary.copy()
fd_plot["pair_label"] = fd_plot["pair"] + "/" + fd_plot["comparison"]
pair_medians = fd_plot.groupby("pair_label")["auc_family_macro"].median().reindex(
    [f"{x}/{y}" for x, y in PAIRS.items()]
)
ax.bar(range(len(pair_medians)), pair_medians.values)
ax.axhline(0.5, linestyle="--", linewidth=0.8)
ax.axhline(0.7, linestyle=":", linewidth=0.8)
ax.set_xticks(range(len(pair_medians)), pair_medians.index, rotation=70, ha="right")
ax.set_ylabel("Median AUC after dropping one X/Y family pair")
ax.set_title("Family-swapped evaluation robustness · Layer 24 · assistant-start · neutral")
fig.tight_layout()
fig.savefig(OUT / "family_swap_robustness.png", dpi=160)
plt.show()
plt.close(fig)


## 8. Reproducibility manifest and export

The ZIP contains the full shuffled AUC distributions, not just summary statistics. The original design files, new held-out sentences, model revision, source hash, and run environment are recorded.

In [ ]:
manifest = {
    "status": "COMPLETED",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_layers": int(model.config.num_hidden_layers),
    "hidden_size": int(model.config.hidden_size),
    "dtype": "float32",
    "attention": getattr(model.config, "_attn_implementation", "unknown"),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "sklearn": sklearn.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "seed": SEED,
    "layers": LAYERS,
    "positions": POSITIONS,
    "conditions": CONDITIONS,
    "concept_pairs": PAIRS,
    "training_rows": int(len(train_df)),
    "evaluation_rows": int(len(eval_df)),
    "unique_prompts": int(len(PROMPTS)),
    "training_families_per_concept": 6,
    "training_wrappers_per_family": 5,
    "heldout_families_per_concept": 5,
    "heldout_wrappers_per_family": 4,
    "training_folds": 6,
    "balanced_label_assignments_per_pair_fold": 252,
    "total_shuffled_fits": int(len(shuffle_df)),
    "prompt_sha256": PROMPT_HASH,
    "training_source_sha256": actual_sha,
    "interpretation": (
        "X-vs-Y relative concept identity classification from hidden states. "
        "Not an absolute target-presence detector, thought measure, or causal test."
    ),
    "caveats": [
        "Repeated wrappers are not independent semantic examples.",
        "Training folds overlap and are sensitivity checks, not independent replications.",
        "Held-out descriptions are author-created; automatic checks do not replace independent human annotation.",
        "The exact shuffle control permutes labels by whole training family.",
        "No multiple-comparison correction is applied across 20 concept pairs.",
        "AUC is a ranking measure and does not establish calibration or causal relevance."
    ]
}
(OUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

# Compact text summary for quick inspection.
summary_lines = [
    "# Qwen Concept Probe Robustness — Run Summary",
    "",
    f"- Model: `{MODEL_ID}`",
    f"- Revision: `{MODEL_REVISION}`",
    f"- Pairs: {len(PAIRS)}",
    f"- Unique prompts: {len(PROMPTS)}",
    f"- Exact shuffled fits: {len(shuffle_df):,}",
    "",
    "## Pair summary",
    "",
    pair_summary_df.to_markdown(index=False),
    "",
    "## Condition / layer / position summary",
    "",
    condition_summary_df.to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "This is an exploratory validation of an X-vs-Y Concept Probe. It does not establish absolute concept activation, thought, or causality."
]
(OUT / "README.md").write_text("\n".join(summary_lines), encoding="utf-8")

zip_path = Path("/content/qwen_concept_probe_20pairs_results.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if p.is_file() and CACHE not in p.parents:
            zf.write(p, arcname=str(p.relative_to(OUT)))
    # Keep feature batches out of the distribution ZIP by default; they are large and reproducible.
    # The manifest and prompt hash are sufficient to document the extraction inputs.

print("Results ZIP:", zip_path)
print("Pair summary:", OUT / "pair_summary.csv")
print("Full shuffle distributions:", OUT / "shuffle_null_full.csv")
print("Manifest:", OUT / "manifest.json")

# Save easy-to-view summaries and the complete ZIP to GitHub. All create-only;
# reruns with identical bytes succeed, while conflicting run IDs fail loudly.
(OUT / "github_result_index.md").write_text(
    f"# Qwen 20-pair Concept Probe results\n\n"
    f"[Open experiment notebook in Colab](https://colab.research.google.com/github/{GITHUB_REPOSITORY}/blob/main/{EXPERIMENT_SLUG}/qwen_concept_probe_robustness_20pairs.ipynb)\n\n"
    f"- Model: `{MODEL_ID}`\n- Revision: `{MODEL_REVISION}`\n"
    f"- Inputs: {N_PROMPTS}\n- Label shuffle fits: {len(shuffle_df)}\n"
    f"- Complete ZIP: `qwen_concept_probe_20pairs_results.zip`\n"
    f"- Human-readable summary: `README.md`\n\n"
    f"Interpretation: exploratory, X-versus-Y relative classification.\n",
    encoding="utf-8"
)
for file_path, destination in [
    (OUT / "github_result_index.md", "README.md"),
    (OUT / "manifest.json", "manifest.json"),
    (OUT / "pair_summary.csv", "pair_summary.csv"),
    (OUT / "condition_layer_position_summary.csv", "condition_layer_position_summary.csv"),
    (OUT / "README.md", "analysis_summary.md"),
    (zip_path, zip_path.name),
]:
    github_save_bytes(
        f"{GITHUB_RUN_PREFIX}/{destination}",
        file_path.read_bytes(),
        f"Save Qwen Probe 20-pair verified results: {destination}"
    )
print("GitHub results:", f"https://github.com/{GITHUB_REPOSITORY}/tree/{GITHUB_BRANCH}/{GITHUB_RUN_PREFIX}")

from google.colab import files
files.download(str(zip_path))
